$\textbf{OPT201 : HELPIQUET Victor, KERJEAN Adrien}$ 


Vous trouverez dans ce fichier python tous les codes que l'on a utilisé pour notre rapport. Pour des raisons de format, certains graphes sont générés ici mais pas présents dans notre rapport écrit. 

$\textbf{Import : }$

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import cvxpy as cp
import scipy.optimize as sp
import scipy
import pandas as pd
import yfinance as yf



#Pour éviter warning incessants
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)

$\huge \textbf{Question 1 : }$

In [ ]:
#Question 1 : 
n = 20
m = 30
np.random.seed(1)
A = np.random.randn(m, n)
b = np.random.randn(m)

x = cp.Variable(n)
objective = cp.Minimize(0.5 * cp.sum_squares(A @ x - b))
constraints = [0 <= x, x <= 1]
prob = cp.Problem(objective, constraints)

# utilisaitono de prob.solve().
result = prob.solve()
# valeur de x opti dans x.value
print(x.value)

$\huge \textbf{Question 2 : }$

In [ ]:
# #Question 2 : 
import time

#On définit une fonction pour résoudre le problème : 
def solve_problem(n, m):
    A = np.random.randn(m, n)
    b = np.random.randn(m)
    x = cp.Variable(n)
    objective = cp.Minimize(0.5 * cp.sum_squares(A @ x - b))
    constraints = [x >= 0, x <= 1]
    problem = cp.Problem(objective, constraints)
    start = time.time()
    problem.solve()
    end = time.time()
    return end - start

x1 = np.linspace(20, 2000, 10) #Le dernier argument de linspace n'est pas le pas, mais le nombre de points entre les bornes
x2 = np.linspace(30, 3000, 10)

times = []
for n, m in zip(x1, x2): #la dépendance en m est prise en compte ici 
    t = solve_problem(int(n), int(m)) #car on attend des entiers pour faire les dimensions des matrices 
    times.append(t)


plt.plot(x1, times)

plt.grid()
plt.legend(fontsize=20)

# Titre des axes
plt.xlabel(r'n', fontsize=20)
plt.ylabel(r'temps de calcul en seconde', fontsize=20)
plt.title('Temps de calcul comme fonction de n et m',  fontsize=18)
# figure en pdf
plt.tight_layout()
plt.savefig('temps_de_calcul.pdf')

$\huge \textbf{Question 3 : }$

La question est commentée car cette cellule prend trop de temps à s'exécuter (plus d'une heure et ça tournait encore). Le code a néanmoins été fait.

In [ ]:
# #Question 3 :

# from scipy.optimize import minimize

# def solve_scipy(n, m):
#     A = np.random.randn(m, n)
#     b = np.random.randn(m) 
#     # def f(x):
#       return 0.5 * np.sum((A @ x - b)**2)  

#     bounds = [(0, 1)] * n  # Contraintes 0 <= x <= 1
#     x0 = np.random.rand(n)  # Point de départ
#     start = time.time()
#     result = minimize(f, x0, bounds=bounds, method='SLSQP', options={'maxiter': 100, 'ftol': 1e-6})
#     end = time.time()
#     return end - start, result.x, A, b

# nlist = np.linspace(20, 2000, 10).astype(int)
# mlist = np.linspace(30, 3000, 10).astype(int)

# times_scipy = []
# solutions_scipy = []

# for n, m in zip(nlist, mlist):
#     t, sol, A, b = solve_scipy(n, m)
#     times_scipy.append(t)
#     solutions_scipy.append(sol)

# plt.plot(nlist, times_scipy, marker='o', label='Scipy SLSQP')
# plt.xlabel('Taille n')
# plt.ylabel('Temps de calcul (s)')
# plt.title('Temps de résolution avec Scipy SLSQP')
# plt.legend()
# plt.grid()
# plt.tight_layout()
# plt.savefig('temps_scipy.pdf')
# plt.show()

$\Huge{\text{Résoudre le problème du portefeuille markovien}}$

$\huge \textbf{Question 4 : }$

In [ ]:
#Fonction dont on va avoir besoin pour télécharger les données : 
def download_finance_data(n_assets=10):

    # Date range
    start = '2016-01-01'
    end = '2019-12-30'

    # Tickers of assets
    assets = ['JCI', 'TGT', 'CMCSA', 'CPB', 'MO', 'MMC', 'JPM',
              'ZION', 'PSA', 'BAX', 'BMY', 'LUV', 'PCAR', 'TXT', 'TMO',
                'MSFT', 'HPQ', 'SEE', 'VZ', 'CNP', 'NI', 'T', 'BA']
    assets.sort()

    # Downloading data
    if n_assets>23:
        print('Warning: max number of assets is limited to 23')
        n_assets = 23

    # Training
    training_data = yf.download(assets[:n_assets], start=start, end=end, group_by="ticker",auto_adjust=True)  #ajout du dernier pour lever les warning

    # Testing
    testing_data = yf.download(assets[:n_assets], start='2020-01-01', end='2020-12-30', group_by="ticker", auto_adjust=True) #ajout du dernier pour lever les warning

    Y = dict()
    # Compute the monthly returns:
    for ast in assets[:n_assets]:
        qq = training_data[ast]['Close']
        Y[ast] = [100*(qq[ii]- qq[ii-1])/qq[ii-1] for ii in range(1,len(qq))]

    training_df = pd.DataFrame(data=Y)

    Y = dict()
    # Compute the monthly returns:
    for ast in assets[:n_assets]:
        qq = testing_data[ast]['Close']
        Y[ast] = [100 * (qq[ii] - qq[ii - 1]) / qq[ii - 1] for ii in range(1, len(qq))]

    testing_df = pd.DataFrame(data=Y)

    return training_df, testing_df

#Compute_moments
def compute_moments(y_data):
    # Defining initial inputs
    mu = y_data.mean().to_numpy().reshape(1, -1)
    sigma = y_data.cov().to_numpy()

    return mu, sigma

In [ ]:
# Téléchargement des données : 
Ytrain, Ytest = download_finance_data(n_assets = 3) #Ici qu'on va changer le nb d'assets
mu, sigma = compute_moments(Ytrain)

#Réponse à la question 4 : Implémentation de markovitz_portfolio

def markovitz_portfolio(mu, sigma) : #attention, ici mu est un tableau 2D, qui est dans la bonne forme (pas besoin de le transposer dans le calcul des constraints)
    rmin = 1.6/252
    n = np.size(mu)
    x = cp.Variable(n)
    objective = cp.Minimize(x.T @ sigma @ x)
    constraints = [mu @ x >= rmin, x >= 0, cp.sum(x) == 1]
    prob = cp.Problem(objective, constraints)

    # La valeur optimale est retournée par `prob.solve()`.
    result = prob.solve()
    # La valeur optimale pour x est rangée dans x.value.
    return(x.value)

x = markovitz_portfolio(mu, sigma)
print(f"La valeur optimale de x est x={x}", type(x))

$\textbf{Commentaires pour comprendre ce qu'on manipule : } $ x représente ici les poids optimaux (allocation du portefeuille) à attribuer à chaque actif dans le portefeuille Markowitz. Chaque colonne correspond à un actif. Chaque ligne de Y correspond à une observation, chaque colonne correpond à un actif

Pour bien comprendre : 
L'objectif est de minimiser la variance (risque) du portefeuille (car souvent, minimiser la variance du portfolio permet d'avoir de bons résultats, mais parfois ne suffit pas)
$$
x^T \sigma x
$$
sous la contrainte que le rendement attendu du portefeuille
$$
\mu^T x
$$
soit au moins égal à un seuil $ r_{\min} $, et que les poids soient positifs et somment à 1 (portefeuille totalement investi, sans ventes à découvert).
mu_i est le rendement de l'actif i dans le portefeuille. x_i est la proportion du portefeuille investie dans i.


$\huge \textbf{Question 5 : }$

In [ ]:
def compute_metrics(x, training_df, testing_df):
    # Calculating Annualized Portfolio Stats

    var0 = pd.DataFrame(x * (training_df.cov() @ x)) #On a ajouté le fait de le retransformer en dataFrame, car soit le jupyter notebook changeait le type, soit il y avait un autre turc, mais ça ne fonctionnait pas.
    var0 = var0.sum().to_frame().T
    std0 = np.sqrt(var0* 252)
    ret0 = training_df.mean().to_frame().T @ x * 252

    var = pd.DataFrame(x * (testing_df.cov() @ x)) #On fait la même chose ici
    var = var.sum().to_frame().T
    std = np.sqrt(var* 252)
    ret = testing_df.mean().to_frame().T @ x * 252


    stats_training  = pd.concat([ret0, std0, var0], axis=0)
    stats_testing = pd.concat([ret, std, var], axis=0)

    #stats = pd.concat([training_metrics, testing_metrics], axis=1)
    stats_training.index = ['Return', 'Std. Dev.', 'Variance']
    stats_testing.index = ['Return', 'Std. Dev.', 'Variance']

    print('Training set: 2016 -- 2019')
    print(stats_training)
    print('Testing set: 2020')
    print(stats_testing)

    return stats_training, stats_testing

In [ ]:
s_tr, s_te = compute_metrics(x, Ytrain, Ytest)

$\huge \textbf{Question 6 :}$

In [ ]:
# Question 6
import scipy

def markovitz_portfolio_probabilistic(mu, sigma,beta=0.49):
    alpha=1.6/252
    n=np.size(mu)
    x = cp.Variable(n)
    objective = cp.Minimize(x@sigma@x)
    constraints = [
    mu@x+scipy.stats.norm.ppf(beta)*cp.sum_squares(cp.sqrt(sigma)@x)>=alpha, 
    x>=0, 
    cp.sum(x) == 1]
    prob = cp.Problem(objective, constraints)

    result = prob.solve()
    # La valeur optimale pour x est stockée dan x.value.
    return(x.value)

In [ ]:
# Analyse the results with the training and testing data. 
Ytrain, Ytest = download_finance_data(n_assets=3)
mu, sigma = compute_moments(Ytrain)

x = markovitz_portfolio_probabilistic(mu, sigma) 
print(x)

def compute_metrics2(x, training_df, testing_df): #on est obligé de la modifier car sinon on a une erreur sur le format en sortie
    cov_train = training_df.cov()
    cov_test = testing_df.cov()

    # Variance scalaire : x^T @ cov @ x
    var0 = float(x.T @ cov_train.to_numpy() @ x)  
    var = float(x.T @ cov_test.to_numpy() @ x)

    std0 = np.sqrt(var0*252)
    std = np.sqrt(var*252)

    ret0 = float(training_df.mean().to_numpy() @ x * 252)
    ret = float(testing_df.mean().to_numpy() @ x * 252)

    stats_training = pd.DataFrame([ret0, std0, var0], index=['Return', 'Std. Dev.', 'Variance'], columns=['Training'])
    stats_testing = pd.DataFrame([ret, std, var], index=['Return', 'Std. Dev.', 'Variance'], columns=['Testing'])

    print('Training set: 2016 -- 2019')
    print(stats_training)

    print('Testing set: 2020')
    print(stats_testing)

    return stats_training, stats_testing

s_tr, s_te = compute_metrics2(x, Ytrain, Ytest)

In [ ]:
# Test pour les données avec 3 actifs et β=0.49
x = markovitz_portfolio_probabilistic(mu, sigma)
print(f"La valeur optimale est x={x}\n")

# Il faut qu'on mette aussi le compute_metrics pour avoir des données qualitatives sur les résultats qui sortent  
# On augmente le nombre d'actifs

variances_train = []
variances_test = []
n_assets_list = range(4, 11)

for i in n_assets_list:
    Ytrain, Ytest = download_finance_data(n_assets=i)
    mu, sigma = compute_moments(Ytrain)
    x = markovitz_portfolio(mu, sigma)
    s_tr, s_te = compute_metrics2(x, Ytrain, Ytest)

    var_train = s_tr.loc['Variance', 'Training']
    var_test  = s_te.loc['Variance', 'Testing']

    variances_train.append(var_train)
    variances_test.append(var_test)

# Deux subplots : train et test
plt.figure(figsize=(10,4))

# Subplot 1 : train
plt.subplot(1, 2, 1)
plt.plot(list(n_assets_list), variances_train, marker='o')
plt.xlabel('Nombre d’actifs', fontsize=20)
plt.ylabel('Variance (train)', fontsize=20)
plt.title('Variance train en fonction du nombre d’actifs',fontsize=15)
plt.grid(True)

# Subplot 2 : test
plt.subplot(1, 2, 2)
plt.plot(list(n_assets_list), variances_test, marker='o', color='orange')
plt.xlabel('Nombre d’actifs', fontsize=20)
plt.ylabel('Variance (test)', fontsize=20)
plt.title('Variance test en fonction du nombre d’actifs', fontsize=15)
plt.grid(True)

plt.tight_layout()
plt.savefig('variance_train_test_fct_nb_assets_Q6.pdf')
plt.show()


# On fait varier la valeur de β, pour une valeur de n fixe 
for i in range(6): #On ajoute un if pour pouvoir descendre en Beta, Sinon, la contrainte devient trop forte, et donc la résolution numérique ne se fait plus. (en pratique, on se s'arrêtait à 0.487)
    Ytrain, Ytest = download_finance_data(n_assets=3)
    mu, sigma = compute_moments(Ytrain)
    x = markovitz_portfolio_probabilistic(mu, sigma,0.49-0.001*i)
    if (x is not None):
        s_tr, s_te = compute_metrics2(x, Ytrain, Ytest)
    print(f"La valeur optimale pour β={0.49-0.001*i} est x={x}\n")

$\huge \textbf{Question 7 :}$

In [ ]:
#On va ajouter les années 2020 et 2021 dans le set de test : ainsi, on va changer la manière dont on download les données : 
def download_finance_databis(n_assets=10):

    # Date range
    start = '2016-01-01'
    end = '2019-12-30'

    # Tickers of assets
    assets = ['JCI', 'TGT', 'CMCSA', 'CPB', 'MO', 'MMC', 'JPM',
              'ZION', 'PSA', 'BAX', 'BMY', 'LUV', 'PCAR', 'TXT', 'TMO',
                'MSFT', 'HPQ', 'SEE', 'VZ', 'CNP', 'NI', 'T', 'BA']
    assets.sort()

    # Downloading data
    if n_assets>23:
        print('Warning: max number of assets is limited to 23')
        n_assets = 23

    # Training
    training_data = yf.download(assets[:n_assets], start=start, end=end, group_by="ticker")

    # Testing : Ici, on a élargi à 2021 dans les données de test.
    testing_data = yf.download(assets[:n_assets], start='2020-01-01', end='2021-12-30', group_by="ticker") 

    Y = dict()
    # Compute the monthly returns:
    for ast in assets[:n_assets]:
        qq = training_data[ast]['Close']
        Y[ast] = [100*(qq[ii]- qq[ii-1])/qq[ii-1] for ii in range(1,len(qq))]

    training_df = pd.DataFrame(data=Y)

    Y = dict()
    # Compute the monthly returns:
    for ast in assets[:n_assets]:
        qq = testing_data[ast]['Close']
        Y[ast] = [100 * (qq[ii] - qq[ii - 1]) / qq[ii - 1] for ii in range(1, len(qq))]

    testing_df = pd.DataFrame(data=Y)

    return training_df, testing_df

In [ ]:
Ytrain, Ytest = download_finance_databis(n_assets=3)
mu, sigma = compute_moments(Ytrain)

#On résout le problème d'optimisation avec ces données maintenant : 
x = markovitz_portfolio_probabilistic(mu, sigma)
print(x)

def compute_metrics2(x, training_df, testing_df): #on est obligé de la modifier car sinon on a une erreur sur le format en sortie
    cov_train = training_df.cov()
    cov_test = testing_df.cov()

    # Variance scalaire : x^T @ cov @ x
    var0 = float(x.T @ cov_train.to_numpy() @ x)  
    var = float(x.T @ cov_test.to_numpy() @ x)

    std0 = np.sqrt(var0*252)
    std = np.sqrt(var*252)

    ret0 = float(training_df.mean().to_numpy() @ x * 252)
    ret = float(testing_df.mean().to_numpy() @ x * 252)

    stats_training = pd.DataFrame([ret0, std0, var0], index=['Return', 'Std. Dev.', 'Variance'], columns=['Training'])
    stats_testing = pd.DataFrame([ret, std, var], index=['Return', 'Std. Dev.', 'Variance'], columns=['Testing'])

    print('Training set: 2016 -- 2019')
    print(stats_training)

    print('Testing set: 2020 - 2021') #Ici, on prend bien les données aussi de 2021 pour faire le test
    print(stats_testing)

    return stats_training, stats_testing

s_tr_with_2021, s_te_with_2021 = compute_metrics2(x, Ytrain, Ytest)

Markovitz-portfolio dans la version probabiliste ou pas est capricieux. Il se peut qu'il renvoie une erreur alors que tout va bien...

In [ ]:
#On va ajouter les années 2020 et 2021 dans le set de test : ainsi, on va changer la manière dont on download les données : 
def download_finance_databis(n_assets=10):

    # Date range
    start = '2016-01-01'
    end = '2019-12-30'

    # Tickers of assets
    assets = ['JCI', 'TGT', 'CMCSA', 'CPB', 'MO', 'MMC', 'JPM',
              'ZION', 'PSA', 'BAX', 'BMY', 'LUV', 'PCAR', 'TXT', 'TMO',
                'MSFT', 'HPQ', 'SEE', 'VZ', 'CNP', 'NI', 'T', 'BA']
    assets.sort()

    # Downloading data
    if n_assets>23:
        print('Warning: max number of assets is limited to 23')
        n_assets = 23

    # Training
    training_data = yf.download(assets[:n_assets], start=start, end=end, group_by="ticker")

    # Testing : Ici, on a élargi à 2021 dans les données de test.
    testing_data = yf.download(assets[:n_assets], start='2021-01-01', end='2021-12-30', group_by="ticker") #on prend ici que l'année 2021, sans l'année 2020

    Y = dict()
    # Compute the monthly returns:
    for ast in assets[:n_assets]:
        qq = training_data[ast]['Close']
        Y[ast] = [100*(qq[ii]- qq[ii-1])/qq[ii-1] for ii in range(1,len(qq))]

    training_df = pd.DataFrame(data=Y)

    Y = dict()
    # Compute the monthly returns:
    for ast in assets[:n_assets]:
        qq = testing_data[ast]['Close']
        Y[ast] = [100 * (qq[ii] - qq[ii - 1]) / qq[ii - 1] for ii in range(1, len(qq))]

    testing_df = pd.DataFrame(data=Y)

    return training_df, testing_df

In [ ]:
Ytrain, Ytest = download_finance_databis(n_assets=3)
mu, sigma = compute_moments(Ytrain)

#On résout le problème d'optimisation avec ces données maintenant : 
x = markovitz_portfolio_probabilistic(mu, sigma)
print(x)

def compute_metrics2(x, training_df, testing_df): #on est obligé de la modifier car sinon on a une erreur sur le format en sortie
    cov_train = training_df.cov()
    cov_test = testing_df.cov()

    # Variance scalaire : x^T @ cov @ x
    var0 = float(x.T @ cov_train.to_numpy() @ x)  
    var = float(x.T @ cov_test.to_numpy() @ x)

    std0 = np.sqrt(var0*252)
    std = np.sqrt(var*252)

    ret0 = float(training_df.mean().to_numpy() @ x * 252)
    ret = float(testing_df.mean().to_numpy() @ x * 252)

    stats_training = pd.DataFrame([ret0, std0, var0], index=['Return', 'Std. Dev.', 'Variance'], columns=['Training'])
    stats_testing = pd.DataFrame([ret, std, var], index=['Return', 'Std. Dev.', 'Variance'], columns=['Testing'])

    print('Training set: 2016 -- 2019')
    print(stats_training)

    print('Testing set: 2021') #Ici, on prend bien les données de 2021 pour faire le test
    print(stats_testing)

    return stats_training, stats_testing

s_tr_with_2021, s_te_with_2021 = compute_metrics2(x, Ytrain, Ytest)

On retrouve bien qu'on a un retour plus élevé que lorsqu'on considère que 2020 comme année de test. La variance est même plus faible pour les données test de 2021 que pour les données d'entrainement, donc il y a beaucoup moins de volatilité, car 2021 ressemble beaucoup plus aux années COVID. Si on avait augmenté le nombre d'assets, on aurait encore eu un meilleur rendement (au-delà de 20%).   

$\huge \textbf{Question 8 :}$

In [ ]:
Ytrain, Ytest = download_finance_data(n_assets=3) # time series #n_assets à changer si on change le nb d'assets à la question 9.

plt.figure(figsize=(10, 5))

# Tracer chaque actif du training set
for asset in Ytrain.columns:
    plt.plot(Ytrain.index, Ytrain[asset].cumsum(), label=f'Train {asset}', linewidth=2)

plt.xlabel('Date', fontsize=23)
plt.ylabel('Valeur', fontsize=23)
plt.title('Train : séries temporelles pour 3 actifs', fontsize=23)
plt.legend()
plt.grid()
plt.tight_layout()
plt.savefig('series_temporelles_Q8_train.pdf')
plt.show()


plt.figure(figsize=(10, 5))
# Tracer chaque actif du test set
for asset in Ytest.columns:
    plt.plot(Ytest.index, Ytest[asset].cumsum(), label=f'Test {asset}')

plt.xlabel('Date', fontsize=23)
plt.ylabel('Valeur', fontsize=23)
plt.title('Test : séries temporelles pour 3 actifs', fontsize=23)
plt.legend()
plt.grid()
plt.tight_layout()
plt.savefig('series_temporelles_Q8_test.pdf')
plt.show()

$\huge \textbf{Question 9}$

In [ ]:
#Fonction donnée pour faire simulation de Monte-Carlo : 
def montecarlo_sim(num_assets,n_samples, Y):
    ####################################
    # Montecarlo Simulation
    ####################################

    # Montecarlo simulation of portfolio weights
    rs = np.random.RandomState(seed=123)
    s1 = rs.dirichlet([0.1] * num_assets, n_samples)
    s2 = rs.dirichlet([0.25] * num_assets, n_samples)
    s3 = rs.dirichlet([0.5] * num_assets, n_samples)
    s4 = rs.dirichlet([0.75] * num_assets, n_samples)
    s5 = rs.dirichlet([1.0] * num_assets, n_samples)
    s6 = rs.dirichlet([1.5] * num_assets, n_samples)
    s7 = rs.dirichlet([2.0] * num_assets, n_samples)
    s8 = rs.dirichlet([3.0] * num_assets, n_samples)
    sample = np.concatenate([np.identity(num_assets), s1, s2, s3, s4, s5, s6, s7, s8], axis=0)

    # Calculating mean, standard deviation and square root kurtosis of each portfolio
    m = sample.shape[0]
    M_1 = np.mean(Y.to_numpy(), axis=0).reshape(1, -1)
    M_2 = Y.cov().to_numpy()

    c_mean = 252 * M_1 @ sample.T
    c_var = np.zeros(m)
    #c_kurt = np.zeros(m)

    for i in range(0, m):
        c_var[i] =  (252 * sample[i] @ M_2 @ sample[i].T) ** (0.5)
        #c_kurt[i] = (np.kron(sample[i], sample[i]) @ Sigma_4 @ np.kron(sample[i], sample[i]).T) ** (1 / 4)

    return c_mean, c_var


#Fonction fournie également pour l'affichage : 

def scatter_plot_port(c_mean, c_var, ret, std, title =""):
    ####################################
    # Plotting Portfolios
    ####################################

    fig, ax = plt.subplots(1, 1, figsize=(12, 10))
    ax = np.ravel(ax)

    # Plotting Portfolios in mean-standard deviation plane
    cax0 = ax[0].scatter(c_var, c_mean, c=c_mean / c_var, cmap='Spectral')
    ax[0].scatter(std,
                  ret,
                  marker='*',
                  s=2 ** 8,
                  color='tab:red',
                  label='Computed solution')

    plt.xlabel('Standard deviation [%]', fontsize = 28)
    plt.ylabel('Return [%]', fontsize = 28)
    if title:
        plt.title(title, fontsize=28)
    plt.grid()
    plt.legend(fontsize=28)
    plt.tight_layout()
    # plt.savefig('montecarlo_3_Q9_train.pdf') # Attention, ici, on aura qu'une figure sur les deux sauvegardée au format pdf. Pour avoir les deux, il faut commenter et décommenter à la main.
    plt.show()

    return

In [ ]:
Ytrain, Ytest = download_finance_data(n_assets=3) # Pour être sûr d'avoir les bonnes données. On peut changer plus facilement le nombre d'assets aussi.

In [ ]:
n_samples = 1000
n_assets = 3
# on change le nombre d'actifs : Attention, quand on change n_assets, il faut aussi le changer dans Y_train pour avoir le bon nb de données correspondant
# n_assets = 20 
# Pour le rendu du code, on ne garde que le n = 3 à l'affichage.
c_mean, c_var = montecarlo_sim(n_assets, n_samples, Ytrain)
scatter_plot_port(c_mean, c_var, s_tr.iloc[0], s_tr.iloc[1], "Analyse Monte-Carlo sur données d'entrainement")
c_mean, c_var = montecarlo_sim(n_assets, n_samples, Ytest)
scatter_plot_port(c_mean, c_var, s_te.iloc[0], s_te.iloc[1], "Analyse Monte-Carlo sur données de test")

In [ ]:
# Cas du nombre d'assets = 20 : 
Ytrain, Ytest = download_finance_data(n_assets=20) 

In [ ]:
# Cas du nombre d'assets = 20 :

n_samples = 1000
n_assets = 20
# on change le nombre d'actifs : Attention, quand on change n_assets, il faut aussi le changer dans Y_train pour avoir le bon nb de données correspondant
# n_assets = 20 
# Pour le rendu du code, on ne garde que le n = 3 à l'affichage.
c_mean, c_var = montecarlo_sim(n_assets, n_samples, Ytrain)
scatter_plot_port(c_mean, c_var, s_tr.iloc[0], s_tr.iloc[1], "Analyse Monte-Carlo sur données d'entrainement")
c_mean, c_var = montecarlo_sim(n_assets, n_samples, Ytest)
scatter_plot_port(c_mean, c_var, s_te.iloc[0], s_te.iloc[1], "Analyse Monte-Carlo sur données de test")

Pour 20 assets, on a une analyse similaire à 3  assets : le retour sur les données de test est très mauvais, avec une haute volatilité, tandis que sur les données d'entrainement, bien que la volatilité soit forte, le retour est plus élevé. Il y a similairement un sur-ajustement sur les données d'entrainement.

In [ ]:
Ytrain, Ytest = download_finance_data(n_assets=3) 

$\Huge{3. Kurtosis}$

$\huge\textbf{Question 10 : }$

Voir le rapport du projet.

$\huge\textbf{Question 11 : }$

1ère étape : utiliser la théorie et faire le développement de Taylor 
2ème étape : résoudre la suite de problème pour delta x, en définissant par récurrence la suite.

Le problème en question a un coût quadratique. On a bien le coût au moins deux fois différentiable et les contraintes au moins une fois différentiables.

In [ ]:
# Question 11
n = 20
m = 30
np.random.seed(1)
A = np.random.randn(m, n)
b = np.random.randn(m)

import time #pour le temps de calcul et la comparaison plus tard 

# on va afficher la norme du résidu, c'est à dire la différence entre la solution et celle qu'on a à l'instant k :
#On utilisera CVXPY pour chaque problème convexe : 
residus = [] #on va garder la valeur du résidu à chaque tour pour le plot ensuite.
violation_contrainte = [] #on regarde à quel points notre solution est admissible 
x = np.random.rand(n) #on part d'un point quelconque comme proposé par la théorie 
#on fixe ensuite un nombre d'itérations : 
nb_iter = 10


t0 = time.perf_counter() #calcul du temps de calcul 
for k in range(nb_iter):
    # Linéarisation de la contrainte quadratique
    grad_f = A.T @ (A @ x - b)  # ∇f(x_k)
    linear_lhs = 2 * x.T # on linéarise cette contrainte quadratique, donc non linéaire, par son développement de Taylor 
    rhs_constraint= 0.5 - np.sum(x**2)
    # Problème quadratique à résoudre à chaque tour 
    dx = cp.Variable(n) # variable de minimisation Δx
    objective = cp.Minimize(grad_f @ dx + 0.5 * cp.sum_squares(A @ dx)) #on garde la fonction cost de base car on a une fonction convexe donc convenable pour résoudre le pb
    constraints = [
        dx >= -x, #contraintes qui conviennent sans rien changer donc on les met telles qu'elles.
        dx <= 1-x,
    ]
    # Contrainte linéarisée (active seulement si violée)
    if rhs_constraint > 0:
        constraints.append(linear_lhs @ dx >= rhs_constraint)

    prob = cp.Problem(objective, constraints) #on résout le problème auquel on est arrivé en linéarisant, grâce à CVXPY
    prob.solve() #on utilise la même méthode que pour le problème simple du début pour résoudre le problème, CVXPY choisit direct le solveur le plus adapté pour résoudre le problème
    
    x_new = x + dx.value 
    res = np.linalg.norm(x_new - x)
    
    # Violations contraintes
    viol_box = max(np.max(-x_new), np.max(x_new - 1))
    viol_norm = max(0, 0.5 - np.sum(x_new**2))
    res_constraint = max(viol_box, viol_norm)
    
    residus.append(res)
    violation_contrainte.append(res_constraint) #on évalue si les contraintes sont violées ou non, notamment la deuxième, par le calcul en faisant la différence
    
    print(f"Iter {k}: Residual={res:.3e}, Constraint Violation={res_constraint:.3e}")
    
    x = x_new
    x = np.clip(x, 0, 1) # projette le vecteur x sur [0,1]

t1 = time.perf_counter()
elapsed_scp = t1 - t0 #calcul du temps de calcul 

# Affichage des courbes de convergence
plt.figure(figsize=(10,4))

plt.subplot(1,2,1)
plt.semilogy(residus, marker='o')
plt.title('Norme du résidu', fontsize=23)
plt.xlabel('Itération', fontsize=20)
plt.ylabel(r'$||x_{k+1} - x_k||$', fontsize=20)
plt.grid(True)

plt.subplot(1,2,2)
plt.semilogy(violation_contrainte, marker='x')
plt.title('Violation de contrainte', fontsize=23)
plt.xlabel('Itération', fontsize=20)
plt.ylabel('max(violations)', fontsize=20)
plt.grid(True)
plt.tight_layout()
plt.savefig('Q11_pb_simple.pdf')
plt.show()

# Vérification contraintes finales
print('\nContraintes finales :')
print(f'0 <= x <= 1  ? -> {np.all((x >= 0) & (x <= 1))}')
print(f'||x||^2 >= 0.5 ? -> {np.linalg.norm(x)**2:.3e}')
print(f'Objectif f(x) = {0.5 * np.sum((A @ x - b)**2):.6f}')
print(f"temps de calcul SCP : {elapsed_scp}")

$\huge\textbf{Question 12 : }$

In [ ]:
#Comparaison avec les autres solveurs : 
# Comparaison avec SCIPY.optimize et le solveur SLSQP : on reprend le même code et on change juste le type de solveur comme au début 
# On s'intéresse ici qu'au csa n = 20 et m = 30 contrairement aux premières questions du projet donc pas besoin linspace par exemple.

n = 20
m = 30
np.random.seed(1)
A = np.random.randn(m, n)
b = np.random.randn(m)

# on va afficher la norme du résidu, c'est à dire la différence entre la solution et celle qu'on a à l'instant k :
#On utilisera CVXPY pour chaque problème convexe : 

from scipy.optimize import minimize
from scipy.optimize import LinearConstraint #pour pouvoir intégrer la contrainte linéarisée dans notre résoution 
import time

def f(x):
    return 0.5 * np.sum((A @ x - b)**2)

cons = ({
    'type': 'ineq', 
    'fun': lambda x: np.linalg.norm(x)**2 - 0.5
},)

bounds = [(0.0, 1.0)] * n  #Contraintes 0 <= x <= 1
x0 = np.random.rand(n)  # Point de départ

t0 = time.perf_counter() #calcul du temps de calcul 

res_slsqp = minimize(
    f,
    x0,
    method='SLSQP',
    bounds=bounds,
    constraints=cons,
    options={'ftol': 1e-9, 'maxiter': 500, 'disp': True}
)

t1 = time.perf_counter()
elapsed_slsqp = t1 - t0 #calcul du temps de calcul 

x_slsqp = res_slsqp.x

print("\nSolution SLSQP :")
print(f"Convergence SLSQP: {res_slsqp.success}, message: {res_slsqp.message}")
print(f"0 <= x <= 1 ? -> {np.all((x_slsqp >= 0) & (x_slsqp <= 1))}")
print(f"||x||^2 >= 0.5 ? -> {np.linalg.norm(x_slsqp)**2:.3e}")
print(f"f(x_SLSQP) = {f(x_slsqp):.6f}")
print(f"Nb d’itérations SLSQP: {res_slsqp.nit}")
print(f"temps de calcul SLSQP: {elapsed_slsqp}")
if (elapsed_slsqp > elapsed_scp) :
    print("Temps de calcul plus long pour la méthode SLSQP")
else :
    print("Temps de calcul plus long pour la méthode SCP")

$\huge\textbf{Question 13 : }$

In [ ]:
#Importation des fonctions fournies dont on a besoin : 

# On charge ici les données pour pouvoir les faire varier dans la question 13 : 
Ytrain, Ytest = download_finance_data(n_assets=3)
mu, sigma = compute_moments(Ytrain)
n_assets = 3

def compute_coefficients(Y, n):
    ####################################
    # Auxiliary functions
    ####################################

    # Function that calculates D_2
    def duplication_matrix(n):
        out = np.zeros((int(n * (n + 1) / 2), n ** 2))
        for j in range(1, n + 1):
            for i in range(j, n + 1):
                u = np.zeros((int(n * (n + 1) / 2), 1))
                u[round((j - 1) * n + i - ((j - 1) * j) / 2) - 1] = 1.0
                E = np.zeros((n, n))
                E[i - 1, j - 1] = 1.0
                E[j - 1, i - 1] = 1.0
                out += u @ E.reshape(-1, 1).T
        return out.T

    # Function that calculates L_2
    def duplication_elimination_matrix(n):
        out = np.zeros((int(n * (n + 1) / 2), n ** 2))
        for j in range(n):
            e_j = np.zeros((1, n))
            e_j[0, j] = 1.0
            for i in range(j, n):
                u = np.zeros((int(n * (n + 1) / 2), 1))
                row = round(j * n + i - ((j + 1) * j) / 2)
                u[row] = 1.0
                e_i = np.zeros((1, n))
                e_i[0, i] = 1.0
                out += np.kron(u, np.kron(e_j, e_i))
        return out

    # Function that calculates S_4
    def kurt_matrix(Y):
        P = Y.to_numpy()
        T, n = P.shape
        mu = np.mean(P, axis=0).reshape(1, -1)
        mu = np.repeat(mu, T, axis=0)
        x = P - mu
        ones = np.ones((1, n))
        z = np.kron(ones, x) * np.kron(x, ones)
        S4 = 1 / T * z.T @ z
        return S4

    return duplication_matrix(n), duplication_elimination_matrix(n), kurt_matrix(Y)

$\textit{Réponse à la question 13 :}$ 

In [ ]:
# SCP kurtosis
import time

# on reprend le cas n=3 comme dans la partie 2 pour pouvoir comparer à markovitz_portfolio
# Coefficients pour la kurtosis
D2, L2, S4 = compute_coefficients(Ytrain, n_assets)
S2 = D2.T @ D2 @ L2
Sigma_4_sqrt = scipy.linalg.sqrtm(S2 @ S4 @ S2.T)

n=n_assets
rmin=1.6/252 

def SCP_kurtosis(x,maxiter,eps):
    x0=x.copy()

    residual_list=[]

    # Condtions initiales
    rmin=1.6/252
    k=0
    X = np.outer(x, x)
    z = L2 @ X.reshape(n*n, 1)
    g = 0.0
    g_list = [g]

    add = np.ones(n)  # Initialisation pour entrer dans la boucle

    mu=Ytrain.mean().to_numpy().reshape(1, -1)

    start_time = time.time() #pour compter le temps
    while (k < maxiter) and (np.linalg.norm(add) > eps):

        dx = cp.Variable(n)
        dX = cp.Variable((n, n), symmetric=True)
        dz = cp.Variable((z.shape[0], 1))
        dg = cp.Variable()

        
        x_col = x.reshape(-1, 1)
        dx_col = cp.reshape(dx, (n, 1))

        objective = cp.Minimize(g + dg)

        constraints = [          
            X + dX == x_col @ x_col.T + x_col @ dx_col.T + dx_col @ x_col.T,
            z + dz == L2 @ cp.reshape(cp.vec(X + dX), (n*n, 1)),
            cp.norm(Sigma_4_sqrt @ (z + dz), 2) <= g + dg,
            cp.sum(x + dx) == 1,
            x + dx >= 0,
            mu @ (x + dx) >= rmin,
            X + dX >> 0
        ]

        prob = cp.Problem(objective, constraints)
        prob.solve()

        dx_val = np.array(dx.value).reshape(-1)
        dX_val = np.array(dX.value)
        dz_val = np.array(dz.value)
        dg_val = float(dg.value)

        add = dx_val
        x = x + dx_val
        X = X + dX_val
        z = z + dz_val
        g = g + dg_val
        
        g_list.append(g)  # on garde l'historique
        residual_list.append(np.linalg.norm(dx_val))
        k += 1

    solve_time=time.time()-start_time
    if (k<maxiter):
        print(f"SCP à convergé en k={k} itérations pour la condition initiale x0={x0}")

    print(f"temps={solve_time:.3f}s")

    return(np.array(x), np.array(g_list))


# Portefeuille min-variance
x=markovitz_portfolio(mu,sigma)
x_minvar = x.flatten()

x_minvar0 = x_minvar.copy()

# Uniforme
x_unif = np.ones(n) / n

# Aléatoire
x_rand = np.random.rand(n)
x_rand /= np.sum(x_rand)

# Lancer SCP pour chaque x0
x_scp_minvar, g_minvar = SCP_kurtosis(x_minvar0,500, 1e-4)
x_scp_unif,   g_unif = SCP_kurtosis(x_unif, 500, 1e-4)
x_scp_rand,   g_rand = SCP_kurtosis(x_rand,500, 1e-4)

print(f"Min-variance finale: g = {g_minvar[-1]:.6f}")
print(f"Uniforme finale:     g = {g_unif[-1]:.6f}")
print(f"Aléatoire finale:    g = {g_rand[-1]:.6f}")
print(f"Meilleure g:         {min(g_minvar[-1], g_unif[-1], g_rand[-1]):.6f}")


import matplotlib.pyplot as plt

plt.figure(figsize=(8,5))
plt.plot(g_minvar, '-o', label='Init min-variance', alpha=0.9)
plt.plot(g_unif, '--x', label='Init uniforme', alpha=0.9)
plt.plot(g_rand, ':s', label='Init aléatoire', alpha=0.9)

plt.xlabel('Itération', fontsize = 23)
plt.ylabel('g', fontsize = 23)
plt.title("Évolution de la mesure de kurtosis (g) au fil des itérations", fontsize = 19)
plt.grid(True)
plt.legend()
plt.savefig('Q13_prem_graphe.pdf')
plt.show()


# Comparaison poids finaux vs min-variance
indices = np.arange(n)
width = 0.35

# plt.subplot(1,2,2)
plt.bar(indices - width/2, x_minvar0, width, label='Min-variance')
plt.bar(indices + width/2, x_scp_minvar, width, label='SCP kurtosis')
plt.xlabel('Actif', fontsize = 20)
plt.ylabel('Poids', fontsize = 20)
plt.title('Poids: min-variance vs SCP kurtosis', fontsize = 21)
plt.legend()
plt.grid(True)
plt.savefig('Q13_deux_graphe.pdf')

$\huge{\text{Question 14 :} }$

In [ ]:
# Code de la question 14 :
# Ici, on va utiliser SCIPY et les fonctions directement intégrées pour résoudre le problème de KURTOSIS :

# Pour pouvoir compter le temps + pour avoir la fonction minimize de scipy et surtout le solveur SLSQP (option de minimize):
import time 
from scipy.optimize import minimize

# On va tout découper pour que ce soit clair : 

# Fonction objectif
def kurtosis_objective(vec):
    x = vec[:n]
    g = vec[-1] #la fonction g est la dernière variable sous le min, d'où la prise de g[-1]
    return g


# Contraintes du pb de Kurtosis :
def kurtosis_constraints(vec):
    x = vec[:n]
    # X_vec = vec[n:n+p]
    g = vec[-1]
    
    #Dictionnaire qui va être pratique ensuite pour accéder aux contraintes (on aura juste à utiliser les clés pour avoir les valeurs)
    cons = {}
    # somme x = 1
    cons['sum_x'] = np.sum(x) - 1.0
    # rendement minimum
    cons['return'] = mu @ x - rmin
    # z = L2 @ vec(X)
    X = np.outer(x, x).flatten()
    z = L2 @ X
    cons['kurtosis'] = np.linalg.norm(Sigma_4_sqrt @ z) - g
    # # X = x x.T
    # X_xxT = np.outer(x, x).flatten()
    # cons['X_eq_xxT'] = np.linalg.norm(X_vec - X_xxT)
    
    return cons #on renvoie un dictionnaire chargé avec toutes les contraintes de notre problème

def SLSQP_kurtosis(x0, max_iter=100): #fonction de résoltution pour pouvoir ensuite itérer sur les mêmes conditions que Q13 et pouvoir comparer donc;

    # Initialisation: x0, X0 = x0 x0^T, z0, g0
    x_init = x0 / np.sum(x0)
    X_init = np.outer(x_init, x_init).flatten()  # vec(X)
    z_init = L2 @ X_init
    g_init = np.linalg.norm(Sigma_4_sqrt @ z_init)
    
    variables0 = np.concatenate([x_init, [g_init]]) # X_init,
    
    # Bounds: x >= 0, g libre 
    bounds = [(0, None)] * n +  [(None, None)] # + [(None, None)] * p 
    
    # Contraintes
    constraints = [ #remplissage du dictionnaire de contraintes
        {'type': 'eq', 'fun': lambda v: kurtosis_constraints(v)['sum_x']},
        {'type': 'ineq', 'fun': lambda v: kurtosis_constraints(v)['return']},
        {'type': 'eq', 'fun': lambda v: kurtosis_constraints(v)['kurtosis']},
        # {'type': 'eq', 'fun': lambda v: kurtosis_constraints(v)['X_eq_xxT']}
    ]
    
    start_time = time.time() #pour compter le temps
    res = minimize( #Utilisation du minimize de scipy, en mettant bien SLSQP comme solveur. 
        kurtosis_objective, variables0, method='SLSQP',
        bounds=bounds, constraints=constraints,
        options={'maxiter': max_iter, 'disp': True}
    )
    #Calcul du temps de calcul :
    solve_time = time.time() - start_time
    
    # On remplit grâce aux résultats qu'on obtient :
    x_slsqp = res.x[:n]
    g_slsqp = res.x[-1]
    n_iter = res.nit
    
    print(f"SLSQP: g={g_slsqp:.3e}, temps={solve_time:.3f}s, itérations={n_iter}")
    
    return x_slsqp, g_slsqp, solve_time, n_iter, res

# Lancer SLSQP pour chaque x0 (comme en Q13)
x_slsqp_minvar, g_slsqp_minvar, t_minvar, it_minvar, res_minvar = SLSQP_kurtosis(x_minvar)
x_slsqp_unif,   g_slsqp_unif,   t_unif,   it_unif,   res_unif   = SLSQP_kurtosis(x_unif)
x_slsqp_rand,   g_slsqp_rand,   t_rand,   it_rand,   res_rand   = SLSQP_kurtosis(x_rand)

In [ ]:
# Comparaison des deux méthodes : 
print("COMPARAISON SCP vs SLSQP") #notre seule comparaison, c'est la Question 13, que l'on va donc utiliser ici.


# Question 13: SCP
g_scp = g_minvar[-1]
print(f"SCP:     g={g_scp:.3e}")

# Question 14 : SLSQP  
g_slsqp = g_slsqp_minvar
print(f"SLSQP:   g={g_slsqp:.3e}")

# utilisation de compute_metrics pour comparer la qualité des solutions obtenues par les deux méthodes (on s'inspire d'avant)
s_scp_train, s_scp_test = compute_metrics(x_scp_minvar, Ytrain, Ytest)
s_slsqp_train, s_slsqp_test = compute_metrics(x_slsqp_minvar, Ytrain, Ytest)

# Afichage pour la comparaison : On a du forcer le fait que s_scp_tr.iloc[0] soit un float par exemple, car il s'agissait sinon d'un élement de Pandas.
ret_scp_train = float(s_scp_train.iloc[0])
std_scp_train = float(s_scp_train.iloc[1])
ret_slsqp_train = float(s_slsqp_train.iloc[0])
std_slsqp_train = float(s_slsqp_train.iloc[1])

ret_scp_te = float(s_scp_test.iloc[0])
std_scp_te = float(s_scp_test.iloc[1])
ret_slsqp_te = float(s_slsqp_test.iloc[0])
std_slsqp_te = float(s_slsqp_test.iloc[1])

print(f"SCP train:  ret={ret_scp_train:.2f}%, std={std_scp_train:.2f}%")
print(f"SLSQP train: ret={ret_slsqp_train:.2f}%, std={std_slsqp_train:.2f}%")
print(f"SCP test:   ret={ret_scp_te:.2f}%, std={std_scp_te:.2f}%")
print(f"SLSQP test: ret={ret_slsqp_te:.2f}%, std={std_slsqp_te:.2f}%")

In [ ]:
## Fonctions d'affichages :


plt.figure(figsize=(12,5))


plt.subplot(1,2,1)
plt.semilogy(g_minvar, '-o', label='SCP (Q13)', linewidth=2)
plt.axhline(g_slsqp_minvar, color='red', linestyle='--', 
           label=f'SLSQP final (g={g_slsqp_minvar:.2e})')
plt.xlabel('Itération', fontsize = 25)
plt.ylabel('g_k', fontsize = 25)
plt.title('Convergence kurtosis', fontsize = 25)
plt.legend()
plt.grid(True)

# Performance train/test
plt.subplot(1,2,2)
methods = ['SCP', 'SLSQP']

# convertir en float : même souci qu'avant, on a bien dû forcer le fait que ce soit des float, car sinon on avait des objets pandas que print ne voulait pas sortir
ret_scp_train  = float(s_scp_train.iloc[0])
std_scp_train  = float(s_scp_train.iloc[1])
ret_slsqp_train = float(s_slsqp_train.iloc[0])
std_slsqp_train = float(s_slsqp_train.iloc[1])

ret_scp_test  = float(s_scp_test.iloc[0])
std_scp_test  = float(s_scp_test.iloc[1])
ret_slsqp_test = float(s_slsqp_test.iloc[0])
std_slsqp_test = float(s_slsqp_test.iloc[1])

ret_train = [ret_scp_train,  ret_slsqp_train]
ret_test = [ret_scp_test,  ret_slsqp_test]
std_train = [std_scp_train,  std_slsqp_train]  
std_test = [std_scp_test,  std_slsqp_test]

x_pos = np.arange(len(methods))
width = 0.35
plt.bar(x_pos - width/2, ret_train, width, label='Return train', alpha=0.8)
plt.bar(x_pos + width/2, ret_test, width, label='Return test', alpha=0.8)

plt.xlabel('Méthode', fontsize = 25)
plt.ylabel('Performance (%)', fontsize = 25)
plt.title('Return train vs test', fontsize = 25)
plt.xticks(x_pos, methods)
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.savefig("question14_comparison.pdf", dpi=300, bbox_inches='tight')
plt.show()

$\huge{\text{Question 15 :}}$

In [ ]:
# #Fonction fournie également pour l'affichage :  Redéfinition pour ne pas confondre dans la sauvegarde des pdf.

def scatter_plot_port(c_mean, c_var, ret, std, title =""):
    ####################################
    # Plotting Portfolios
    ####################################

    fig, ax = plt.subplots(1, 1, figsize=(12, 10))
    ax = np.ravel(ax)

    # Plotting Portfolios in mean-standard deviation plane
    cax0 = ax[0].scatter(c_var, c_mean, c=c_mean / c_var, cmap='Spectral')
    ax[0].scatter(std,
                  ret,
                  marker='*',
                  s=2 ** 8,
                  color='tab:red',
                  label='Computed solution')

    plt.xlabel('Standard deviation [%]', fontsize = 23)
    plt.ylabel('Return [%]', fontsize = 23)
    if title:
        plt.title(title, fontsize=20)
    plt.grid()
    plt.legend(fontsize=20)
    plt.tight_layout()
    plt.savefig('montecarlo_3_Q15_slsqp_test.pdf')
    plt.show()

    return

In [ ]:
# Affichage grâce à MonteCarlo et Pie Charts : on va s'appuyer sur ce qu'on a fait avant : 
# On réutilise les fonctions de la q9 qui sont donc déjà déclarées : 

Ytrain, Ytest = download_finance_data(n_assets=3) # Pour être sûr d'avoir les bonnes données. On peut changer plus facilement le nombre d'assets aussi.

n_samples = 1000
n_assets = 3

# Données d'entrainement :
c_mean, c_var = montecarlo_sim(n_assets, n_samples, Ytrain) # Simulation de MonteCarlo
scatter_plot_port(c_mean, c_var, s_scp_train.iloc[0], s_scp_train.iloc[1], "plot SCP_train") #plot pour le scptrain
scatter_plot_port(c_mean, c_var, s_slsqp_train.iloc[0], s_scp_train.iloc[1], "plot SLSQP_Train") #plot pour le slsqptrain


# Données de test :
c_mean, c_var = montecarlo_sim(n_assets, n_samples, Ytest) #simulation de Monte Carlo
scatter_plot_port(c_mean, c_var, s_scp_test.iloc[0], s_scp_test.iloc[1], "plot SCP_test") #plot pour le scptest
scatter_plot_port(c_mean, c_var, s_scp_test.iloc[0], s_scp_test.iloc[1], "plot SLSQP_test") #plot pour le scptest

In [ ]:
# Plot qui se superposent (utile pour la comparaison)

def scatter_plot_two_methods(c_mean, c_var,
                             ret1, std1, label1,
                             ret2, std2, label2,
                             title=""):


    fig, ax = plt.subplots(1, 1, figsize=(12, 10))
    ax = np.ravel(ax)

    # Nuage Monte Carlo (std en x, mean en y)
    ax[0].scatter(c_var, c_mean, c=c_mean / c_var, cmap='Spectral')

    # Méthode 1 : étoile rouge
    ax[0].scatter(std1, ret1,
                  marker='*', s=2**8,
                  color='tab:red', label=label1)

    # Méthode 2 : étoile rouge foncé (ou autre nuance)
    ax[0].scatter(std2, ret2,
                  marker='+', s=2**8,
                  color='blue', label=label2)

    ax[0].set_xlabel('Standard deviation [%]', fontsize=30)
    ax[0].set_ylabel('Return [%]', fontsize=30)
    if title:
        ax[0].set_title(title, fontsize=30)
    ax[0].grid(True)
    ax[0].legend(fontsize=30)
    plt.tight_layout()
    # plt.savefig('montecarlo_3_Q15_comparaison_test.pdf')
    plt.show()

n_samples = 1000
n_assets = 3

# Monte Carlo sur TRAIN
c_mean_tr, c_var_tr = montecarlo_sim(n_assets, n_samples, Ytrain)

scatter_plot_two_methods(
    c_mean_tr, c_var_tr,
    s_scp_train.iloc[0],   # return méthode SCP
    s_scp_train.iloc[1],   # std méthode SCP
    "SCP (CVXPY) train",
    s_slsqp_train.iloc[0], # return méthode SLSQP
    s_slsqp_train.iloc[1], # std méthode SLSQP
    "SCP (CVXPY) train", 
    title="Monte Carlo – données d'entraînement"
)

# Monte Carlo sur TEST
c_mean_te, c_var_te = montecarlo_sim(n_assets, n_samples, Ytest)

scatter_plot_two_methods(
    c_mean_te, c_var_te,
    s_scp_test.iloc[0],
    s_scp_test.iloc[1],
    "SCP (CVXPY) test",
    s_slsqp_test.iloc[0],
    s_slsqp_test.iloc[1],
    "SLSQP (SCIPY) test",
    title="Monte Carlo – données de test"
)

On remarque qu'elles se superposent. On s'y attendait.

$\huge{\text{Question 16}: }$

Voir le rapport du projet.

$\huge{\text{Question 17 :}}$

In [ ]:
# On implémente une fonction résolvant le problème (10) pour une nombre d'assets n_assets donné.
# On s'inspire de la fonction solve_perturbation de Q13

import time

def solve_10(n_assets):
    n=n_assets
    Ytrain, Ytest = download_finance_data(n_assets) # téléchargement des données avec un nombre d'assets différent

    # Moments (moyenne et covariance)
    mu, sigma = compute_moments(Ytrain)   # mu: shape (1, n)
    mu = mu.flatten()                     # pour avoir un vecteur 1D de taille n

    rmin = 1.6 / 252

    # Coefficients pour la kurtosis
    D2, L2, S4 = compute_coefficients(Ytrain, n_assets)
    S2 = D2.T @ D2 @ L2
    Sigma_4 = S2 @ S4 @ S2.T
    Sigma_4_sqrt = scipy.linalg.sqrtm(Sigma_4)

    p=n*(n+1)//2

    #####################
    # Problème de perturbation
    #####################


    # Variables CVXPY
    x = cp.Variable(n)
    X = cp.Variable((n, n), symmetric=True)
    z = cp.Variable(p)
    g = cp.Variable()

    # Linéarisation de X_X.T
    block_matrix = cp.bmat([[X, cp.reshape(x, (n, 1))],[cp.reshape(x, (1, n)), np.ones((1, 1))]])

    # Contraintes

    constraints = []
    constraints.append(block_matrix>> 0)
    constraints.append(z == L2 @ cp.vec(X))
    constraints.append(cp.SOC(g, Sigma_4_sqrt @ z))
    constraints.append(cp.sum(x) == 1)
    constraints.append(x >= 0)
    constraints.append(mu @ x >= rmin)

    objective = cp.Minimize(g)
    prob = cp.Problem(objective, constraints)
    start_time=time.time()
    result=prob.solve()
    solve_time=time.time()-start_time

    if prob.status != "optimal":
        print("Perturbation: status =", prob.status)
        return None, None, None, None, None

    return x.value, X.value, g.value,solve_time

x_3, X_3, g_3, time3= solve_10(3)
x_10, X_10, g_10, time10= solve_10(10)
x_23, X_23, g_23, time23= solve_10(23) # nombre d'assets limité à 23


# Pour n_assets=3
print("Pour n_assets = 3")
print("Poids optimaux (x):", x_3)
print("Kurtosis optimisée (g*):", g_3)
print("Erreur ||X - xx^T||:", np.linalg.norm(X_3 - np.outer(x_3, x_3), 'fro'))
print("Temps=",time3)

mean_return_test = np.dot(np.mean(Ytest, axis=0), x_3)
variance_test = np.dot(x_3, np.dot(np.cov(Ytest, rowvar=False), x_3))
print("Retour moyen (test):", mean_return_test)
print("Variance (test):", variance_test)
print("\n")

# Pour n_assets=10
print("Pour n_assets = 10")
print("Poids optimaux (x):", x_10)
print("Kurtosis optimisée (g*):", g_10)
print("Erreur ||X - xx^T||:", np.linalg.norm(X_10 - np.outer(x_10, x_10), 'fro'))
print("Temps=",time10)
print("\n")


# Pour n_assets=23
print("Pour n_assets = 23")
print("Poids optimaux (x):", x_23)
print("Kurtosis optimisée (g*):", g_23)
print("Erreur ||X - xx^T||:", np.linalg.norm(X_23 - np.outer(x_23, x_23), 'fro'))
print("Temps=",time23)
print("\n")

In [ ]:
# Comparaison des résultats entre sur les données de test et d'apprentissage

def stats_portfolio(Y, x):
    r = Y @ x
    mean_ret = np.mean(r)
    var_ret = np.var(r)
    return mean_ret, var_ret

for n_assets in [3, 10, 23]: # 23 est le max d'assets présent dans Y_train 
    # données
    Ytrain, Ytest = download_finance_data(n_assets)

    # min-variance sur Ytrain
    mu, sigma = compute_moments(Ytrain)
    mu = mu.flatten()
    x_mv = markovitz_portfolio(mu, sigma).flatten() 

    # problème (10) sur Ytrain
    x_kur, X_kur, g_kur, time_kur = solve_10(n_assets)

    # stats train
    mean_mv_tr, var_mv_tr = stats_portfolio(Ytrain, x_mv)
    mean_kur_tr, var_kur_tr = stats_portfolio(Ytrain, x_kur)

    # erreur de relaxation
    err_relax = np.linalg.norm(X_kur - np.outer(x_kur, x_kur), 'fro')

    print(f"\n=== n_assets = {n_assets} ===")
    print("Min-variance :  mean_tr =", mean_mv_tr, ", var_tr =", var_mv_tr)
    print("Kurtosis problème (10): mean_tr =", mean_kur_tr, ", var_tr =", var_kur_tr)

$\huge \text{Question 18 :}$ 

Voir le rapport du projet.